# NPHCDA Zero-Dose Platform: External Cross-Checks (Triangulation)

Reproducible analysis for the two convergent-validity cross-checks presented to GAVI, UNICEF and WHO.

1. **Zone level** - our forecast zone zero-dose vs an independent published estimate (Umar et al., *Vaccines* 2025).
2. **LGA level** - our 2026 LGA estimate (as DTP1 coverage) vs IHME modelled LGA DTP1 coverage (admin-2, 2018).

Runs in Google Colab with no setup. Cross-check 1 is fully self-contained (data embedded). Cross-check 2
reads two CSV files you upload: the IHME admin-2 DTP1 file and our ranked LGA export
(`D5_lga_zero_dose_ranked_CLEAN.csv`).

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
NAVY, GREEN, GOLD, RED, GREY = "#1F3B57", "#1C7A3D", "#C8902A", "#C0392B", "#5B6B79"
print("pandas", pd.__version__)

## Cross-check 1 - Zone level vs an independent estimate (Umar et al., 2025)

Our 37-state 2026 forecast is aggregated to the six geopolitical zones (cohort-weighted), then compared
with the zone zero-dose estimates published independently by Umar et al. (geometric-mean zero-dose by
zone; NDHS 2018 + NICS/MICS 2021 + NDHS 2023-24). We test whether the two RANK the zones the same way
(Spearman rho).

In [ ]:
# Our 37-state forecast: (state, zone, observed 2024 %, 2026 mean %, ZD 2026 count)
ROWS = [
 ("Kano","NW",42.4,36.4,194932),("Sokoto","NW",86.5,71.9,168837),("Katsina","NW",39.8,38.1,165813),
 ("Zamfara","NW",82.6,73.1,156169),("Kebbi","NW",84.0,68.3,154105),("Kaduna","NW",45.6,43.4,138383),
 ("Bauchi","NE",36.9,34.3,112564),("Niger","NC",56.0,45.2,104498),("Jigawa","NW",34.3,33.9,103403),
 ("Borno","NE",31.7,30.1,54437),("Benue","NC",32.7,28.9,51888),("Gombe","NE",34.0,34.8,50972),
 ("Kogi","NC",57.0,32.2,48946),("Kwara","NC",51.7,37.4,47028),("Oyo","SW",29.9,19.7,45908),
 ("Yobe","NE",40.4,35.1,42426),("Plateau","NC",35.3,26.1,38814),("Adamawa","NE",29.1,21.1,36734),
 ("Taraba","NE",40.7,27.0,33042),("Rivers","SS",22.7,15.6,31393),("Ogun","SW",20.6,18.2,28254),
 ("Nasarawa","NC",20.3,26.0,27394),("Ondo","SW",23.5,18.5,27319),("Lagos","SW",4.7,7.4,22277),
 ("Delta","SS",7.6,12.8,21658),("Anambra","SE",14.9,9.9,18033),("FCT","NC",6.2,17.1,16538),
 ("Akwa Ibom","SS",15.1,13.0,16453),("Enugu","SE",12.8,10.1,15042),("Edo","SS",3.7,9.5,14067),
 ("Imo","SE",5.6,8.7,14061),("Osun","SW",9.4,11.4,13561),("Cross River","SS",2.8,12.2,13166),
 ("Abia","SE",4.2,8.7,12174),("Ekiti","SW",6.0,10.7,11936),("Bayelsa","SS",19.1,15.1,10423),
 ("Ebonyi","SE",2.6,6.8,7721)]
ZONES = {"NW":"North-West","NE":"North-East","NC":"North-Central","SW":"South-West","SS":"South-South","SE":"South-East"}
EXT = {"North-West":47.4,"North-Central":26.6,"North-East":25.0,"South-West":10.2,"South-South":6.2,"South-East":4.2}

df = pd.DataFrame(ROWS, columns=["state","z","obs24","m26","zd26"])
df["zone"] = df["z"].map(ZONES)
df["cohort"] = df["zd26"] / (df["m26"]/100.0)
rows=[]
for z in df["zone"].unique():
    d=df[df["zone"]==z]
    rows.append((z, d["zd26"].sum()/d["cohort"].sum()*100, (d["cohort"]*d["obs24"]/100).sum()/d["cohort"].sum()*100))
g=pd.DataFrame(rows,columns=["zone","ours_2026","ours_2024"]).set_index("zone")
order=["North-West","North-Central","North-East","South-West","South-South","South-East"]
g=g.reindex(order); g["independent_2024"]=[EXT[z] for z in order]
print(g.round(1))
rho,p=spearmanr(g["ours_2026"],g["independent_2024"])
print("\nSpearman rho (our 2026 vs independent 2024): %.3f  (p=%.2g)"%(rho,p))

In [ ]:
fig,(a,b)=plt.subplots(1,2,figsize=(13,5.5))
zo=sorted(order,key=lambda z:-EXT[z]); x=np.arange(len(zo)); w=0.38
a.bar(x-w/2,[g.loc[z,"ours_2026"] for z in zo],w,label="Our model (2026 forecast)",color=NAVY)
a.bar(x+w/2,[EXT[z] for z in zo],w,label="Independent (Umar et al., 2025; 2024)",color=GREEN)
a.set_xticks(x); a.set_xticklabels([z.replace("-","-\n") for z in zo],fontsize=9)
a.set_ylabel("Zero-dose prevalence (%)"); a.legend(frameon=False,fontsize=8); a.set_title("Zone zero-dose")
for s in ["top","right"]: a.spines[s].set_visible(False)
b.scatter([EXT[z] for z in order],[g.loc[z,"ours_2026"] for z in order],s=70,color=GOLD,edgecolor=NAVY,zorder=3)
b.plot([0,50],[0,50],"--",color=GREY); b.set_xlim(0,50); b.set_ylim(0,50)
b.set_xlabel("Independent estimate, 2024 (%)"); b.set_ylabel("Our model, 2026 forecast (%)")
b.set_title("Rank agreement  -  Spearman rho = %.2f"%rho)
for s in ["top","right"]: b.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## Cross-check 2 - LGA level vs IHME DTP1 coverage (admin-2, 2018)

We convert our LGA zero-dose to DTP1 coverage (coverage = 100 - zero-dose) and compare with IHME's
modelled LGA DTP1 coverage. Because IHME's latest year is 2018 and ours is 2026, we compare on
**rank/tercile** (vintage-robust), not on absolute levels. Each LGA is flagged high / moderate / low
confidence by whether the two sources place it in the same coverage tercile.

**To run:** execute the next cell and upload `IHME_NGA_DTP1_2000_2018_ADMIN_2_*.CSV` and
`D5_lga_zero_dose_ranked_CLEAN.csv` when prompted.

In [ ]:
import re, glob, difflib
def nstate(s):
    s=str(s).lower().strip()
    if "federal capital" in s or s in ("fct","fct, abuja"): return "fct"
    return re.sub(r"\(.*?\)","",s).strip()
def nlga(s):
    s=re.sub(r"\(.*?\)|\[.*?\]","",str(s).lower().strip())
    return re.sub(r"\s+"," ",s.replace("'","").replace("/"," ").replace("-"," ").replace(".","")).strip()
STATE_ALIAS={"nassarawa":"nasarawa"}                       # IHME spells Nasarawa with a double s
LGA_ALIAS={("kebbi","arewa"):"arewa dandi",("imo","ezinihitte mbaise"):"ezinihitte"}
IHME_LGA_ALIAS={("ogun","yewa north"):"egbadonorth",("ogun","yewa south"):"egbadosouth",
                ("lagos","lagos mainland"):"mainland",("ekiti","aiyekire"):"gboyin",
                ("kogi","kogi"):"kotonkar",("abia","obi nwga"):"oboma ngwa"}   # pre-rename names
FUZZY_MIN=0.70

def find_file(must):
    for p in glob.glob("*.CSV")+glob.glob("*.csv"):
        if all(t.lower() in p.lower() for t in must): return p
    return None
ihme_path=find_file(["dtp1","admin_2"]); ours_path=find_file(["clean"])
if ihme_path is None or ours_path is None:
    try:
        from google.colab import files; files.upload()
        ihme_path=find_file(["dtp1","admin_2"]); ours_path=find_file(["clean"])
    except Exception:
        print("Place the two CSV files next to this notebook.")
print("IHME file:",ihme_path,"| our file:",ours_path)

In [ ]:
ours=pd.read_csv(ours_path)
ours["sk"]=ours["State"].map(nstate); ours["lk"]=ours["LGA"].map(nlga)
ours["our_dtp1"]=100.0-pd.to_numeric(ours["ZD proxy (%)"],errors="coerce")
ih=pd.read_csv(ihme_path); ih=ih[ih["year"]==2018].copy()
ih["sk"]=ih["ADM1_NAME"].map(nstate).replace(STATE_ALIAS); ih["lk"]=ih["ADM2_NAME"].map(nlga)
ih["ihme_dtp1"]=pd.to_numeric(ih["mean"],errors="coerce")*100.0
look={(r["sk"],r["lk"]):r["ihme_dtp1"] for _,r in ih.iterrows()}
bystate={}
for _,r in ih.iterrows(): bystate.setdefault(r["sk"],[]).append((r["lk"],r["ihme_dtp1"]))
def match(sk,lk):                                          # exact -> alias -> prefix -> fuzzy
    if (sk,lk) in look: return look[(sk,lk)]
    for amap in (IHME_LGA_ALIAS, LGA_ALIAS):
        al=amap.get((sk,lk))
        if al and (sk,al) in look: return look[(sk,al)]
    cands=bystate.get(sk,[])
    if not cands: return np.nan
    ns=lk.replace(" ","")
    pre=[v for n,v in cands if len(n.replace(" ",""))>=5 and (ns.startswith(n.replace(" ","")) or n.replace(" ","").startswith(ns))]
    if len(pre)==1: return pre[0]
    sc=sorted(((difflib.SequenceMatcher(None,lk,n).ratio(),v) for n,v in cands),key=lambda t:-t[0])
    if sc and sc[0][0]>=FUZZY_MIN and (len(sc)==1 or sc[0][0]-sc[1][0]>=0.05): return sc[0][1]
    return np.nan
ours["ihme_dtp1"]=[match(s,l) for s,l in zip(ours["sk"],ours["lk"])]
m=ours.dropna(subset=["ihme_dtp1","our_dtp1"]).copy()
print("Matched %d of %d LGAs (%.0f%%)"%(len(m),len(ours),len(m)/len(ours)*100))
rho,p=spearmanr(m["our_dtp1"],m["ihme_dtp1"]); print("Spearman rho (our 2026 vs IHME 2018): %.3f (p=%.2g)"%(rho,p))
lab=["Lower third (lowest coverage)","Middle third","Upper third (highest coverage)"]
m["our_t"]=pd.qcut(m["our_dtp1"].rank(method="first"),3,labels=lab)
m["ihme_t"]=pd.qcut(m["ihme_dtp1"].rank(method="first"),3,labels=lab)
d=(m["our_t"].cat.codes-m["ihme_t"].cat.codes).abs()
m["confidence"]=np.select([d==0,d==1],["High","Moderate"],"Low")
print(m["confidence"].value_counts())
m.to_csv("lga_concordance_ihme.csv",index=False); print("Saved lga_concordance_ihme.csv")

In [ ]:
fig,(a,b)=plt.subplots(1,2,figsize=(13,5.5))
ct=pd.crosstab(m["our_t"],m["ihme_t"]).reindex(index=lab,columns=lab).fillna(0).astype(int)
for i in range(3):
    for j in range(3):
        dd=abs(i-j); c=GREEN if dd==0 else (GOLD if dd==1 else RED)
        a.add_patch(plt.Rectangle((j-0.5,i-0.5),1,1,facecolor=c,alpha=0.18,edgecolor="white"))
        a.text(j,i,str(ct.values[i,j]),ha="center",va="center",fontsize=15,fontweight="bold")
a.set_xticks(range(3)); a.set_yticks(range(3))
a.set_xticklabels(["Lower","Middle","Upper"]); a.set_yticklabels(["Lower","Middle","Upper"])
a.set_xlabel("IHME DTP1 tercile (2018)"); a.set_ylabel("Our DTP1 tercile (2026)")
a.set_title("Tercile concordance (green=agree, red=opposite)"); a.set_xlim(-0.5,2.5); a.set_ylim(2.5,-0.5)
b.scatter(m["ihme_dtp1"],m["our_dtp1"],s=12,color=GOLD,alpha=0.6); b.plot([0,100],[0,100],"--",color=GREY)
b.set_xlim(0,100); b.set_ylim(0,100); b.set_xlabel("IHME DTP1 coverage, 2018 (%)"); b.set_ylabel("Our DTP1 coverage, 2026 (%)")
b.set_title("Rank agreement across %d LGAs  -  Spearman rho = %.2f"%(len(m),rho))
for s in ["top","right"]: b.spines[s].set_visible(False)
plt.tight_layout(); plt.show()